# CAP 6776 — Live demo + five-query evaluation (recording notebook)

Use this on **Colab (Runtime → Change runtime type → T4 GPU)** or locally. In order:

1. Install and fetch NFCorpus (`scripts/audit_dataset.py`).
2. Run the unit tests (show in the video that the system is tested).
3. **Five judged queries** → `results/tables/five_query_eval.md` (P@10, nDCG@10, MRR@10, cross-checked with `pytrec_eval`).
4. Launch the **live Gradio demo** (`apps/course_demo.py --share`) and open the `*.gradio.live` link.

Nothing here reads precomputed rankings for the live demo — every query is answered by BM25 / MedCPT / RRF / cross-encoder running now.

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('biomedical-hybrid-ir'):
        subprocess.run(['git', 'clone', 'https://github.com/Arungharami/biomedical-hybrid-ir'], check=True)
    os.chdir('biomedical-hybrid-ir')
    # Colab already ships torch/transformers; add what it lacks.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu', 'pytrec-eval-terrier',
                    'datasets', 'gradio', 'pyyaml', 'nbformat'], check=True)
sys.path.insert(0, os.path.abspath('src'))
import biomedical_ir  # noqa: F401  (fails fast if the checkout is wrong)
print('cwd:', os.getcwd())

In [ ]:
# 1. Download + validate NFCorpus (writes data/raw/nfcorpus and data/processed/*.json)
!python scripts/audit_dataset.py

In [ ]:
# 2. Unit tests for the parts the demo uses (BM25 maths, RRF, metrics, live engine)
!python -m pytest -q tests/test_bm25.py tests/test_fusion.py tests/test_metrics.py tests/test_live_search.py

In [ ]:
# 3. Five judged queries against ALL official test qrels for each query
!python scripts/evaluate_five.py

### 4. Live demo
The first MedCPT query downloads the encoders and embeds 3,633 articles (cached to `data/cache/`).
Click **Warm up models** in the app before you start recording. Stop this cell to shut the app down.

In [ ]:
!python apps/course_demo.py --share